# Inference-Based Figures

This notebook reproduces figures that require model inference:
- **Figure 1**: Spatial transfer success + length scaling failure (bar chart)
- **Table 1**: Composition analysis of length scaling failure
- **Table 6 / Fig 17-20**: Failure case analysis & visualization

**Requirements**: GPU, models in `models/` directory

In [ ]:
import os, sys, pickle, ast
import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Project root
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'evaluation'))

from failure_plotting_utils import (
    load_model_for_inference, 
    collect_failure_cases_from_model,
    annotate_failure_cases,
    visualize_failure_case
)
print('Imports OK')

## Load map data and test pairs

In [ ]:
import random
import networkx as nx
from itertools import product
from collections import defaultdict
from transformers import PreTrainedTokenizerFast

dataset_dir = os.path.join(ROOT, 'dataset')
model_dir = os.path.join(ROOT, 'models')

with open(os.path.join(dataset_dir, 'map_stats', 'nodes_to_indices.pkl'), 'rb') as f:
    nodes_to_indices = pickle.load(f)
with open(os.path.join(dataset_dir, 'map_stats', 'indices_to_nodes.pkl'), 'rb') as f:
    indices_to_nodes = pickle.load(f)
adj_matrix = np.load(os.path.join(dataset_dir, 'map_stats', 'adj_matrix.npy'))

indices_to_nodes = {k: str(v) for k, v in indices_to_nodes.items()}
nodes_to_indices = {str(k): v for k, v in nodes_to_indices.items()}

# ------------------------------------------------------------------------
# Build G1 and G2 test pairs on the fly (same logic as valid_rate.py)
#   G1 = holdout nodes in the TRAINING map (for "Same Map, Longer Length")
#   G2 = disjoint DISJOINT map (for "New Maps, Seen/Longer Length")
# ------------------------------------------------------------------------
G = nx.from_numpy_array(adj_matrix)
hf_tokenizer = PreTrainedTokenizerFast.from_pretrained(os.path.join(model_dir, 'hf_tokenizer'))

def build_test_pairs(indices):
    pairs = []
    for i, j in product(indices, indices):
        if i < j:
            pairs.append({
                'start_index': i,
                'end_index': j,
                'coord_distance': np.linalg.norm(
                    np.array(ast.literal_eval(indices_to_nodes[i])) -
                    np.array(ast.literal_eval(indices_to_nodes[j]))),
                'shortest_path_length': nx.shortest_path_length(G, i, j),
                'prompt': f'<s> {i} {j} :',
                'input_ids': hf_tokenizer.encode(f'<s> {i} {j} :', add_special_tokens=False),
            })
    return pairs

def bucket_by_length(pairs):
    buckets = defaultdict(list)
    intervals = [(1,10),(10,20),(20,30),(30,40),(40,50),(50,60),(60,70),(70,80),(80,100)]
    for r in pairs:
        for a, b in intervals:
            if a <= r['shortest_path_length'] < b:
                buckets[(a, b)].append(r)
                break
    return buckets

# G1: holdout nodes in training map (400 nodes)
test_indices = pickle.load(open(os.path.join(dataset_dir, '_spatial_length', 'test_indices.pkl'), 'rb'))
print(f'Building G1 test pairs from {len(test_indices)} holdout nodes...')
G1_test_pairs = build_test_pairs(test_indices)
G1_length_buckets = bucket_by_length(G1_test_pairs)

# G2: disjoint map — subsample since 2000 nodes => 2M pairs is too many
G2_indices = pickle.load(open(os.path.join(dataset_dir, 'map_stats', 'G2_indices.pkl'), 'rb'))
random.seed(42)
G2_test_indices = random.sample(G2_indices, int(len(G2_indices) * 0.1))
print(f'Building G2 test pairs from {len(G2_test_indices)} sampled disjoint-map nodes...')
G2_test_pairs = build_test_pairs(G2_test_indices)
G2_length_buckets = bucket_by_length(G2_test_pairs)

# Cap each bucket at 3000 pairs for tractable eval
for buckets in [G1_length_buckets, G2_length_buckets]:
    for k in buckets:
        buckets[k] = buckets[k][:3000]

print(f'\nMap nodes: {len(nodes_to_indices)}')
print('G1 (same map holdout) length groups:')
for k, v in sorted(G1_length_buckets.items()):
    print(f'  {k}: {len(v)} pairs')
print('G2 (disjoint map) length groups:')
for k, v in sorted(G2_length_buckets.items()):
    print(f'  {k}: {len(v)} pairs')

## Load best SFT model

In [ ]:
coverage_max = 0.6
diversity_max = 64
pairs_idx = 0

sft_model, sft_tokenizer, sft_hf_tokenizer = load_model_for_inference(
    mode='sft',
    coverage=coverage_max,
    diversity=diversity_max,
    pairs_idx=pairs_idx,
    model_dir=model_dir,
    indices_to_nodes=indices_to_nodes,
    nodes_to_indices=nodes_to_indices,
    adj_matrix=adj_matrix
)
print('SFT model loaded')

## Figure 1: Spatial Transfer + Length Scaling

Evaluate the best SFT model on test pairs grouped by path length.
- Blue bars: within training length (spatial transfer to unseen pairs)
- Red/Orange bars: longer than training length (length scaling)

In [ ]:
# Evaluate on each length group
# G2 (new maps): all length groups — this is the spatial transfer test
# G1 (same map): only longer-length groups — no spatial transfer, just length scaling
max_train_length = 20

g1_results = {}  # Same map, longer length
g2_results = {}  # New maps, all lengths

for length_range in sorted(G2_length_buckets.keys()):
    # G2: New maps (spatial transfer)
    pairs_g2 = G2_length_buckets.get(length_range, [])
    if pairs_g2:
        n = min(len(pairs_g2), 3000)
        pairs_g2 = pairs_g2[:n]
        print(f'\nG2 (new maps) {length_range} ({n} pairs)...')
        sl = annotate_failure_cases(sft_model, sft_tokenizer, sft_hf_tokenizer, 'sft',
                                    pairs_g2, nodes_to_indices, indices_to_nodes, adj_matrix, batch_size=64)
        g2_results[length_range] = sum(sl) / len(sl)
        print(f'  SR = {g2_results[length_range]:.3f}')

for length_range in sorted(G1_length_buckets.keys()):
    if length_range[1] <= max_train_length:
        continue  # Skip ID-length for same map (only need longer)
    # G1: Same map, longer length (no spatial transfer)
    pairs_g1 = G1_length_buckets.get(length_range, [])
    if pairs_g1:
        n = min(len(pairs_g1), 3000)
        pairs_g1 = pairs_g1[:n]
        print(f'\nG1 (same map) {length_range} ({n} pairs)...')
        sl = annotate_failure_cases(sft_model, sft_tokenizer, sft_hf_tokenizer, 'sft',
                                    pairs_g1, nodes_to_indices, indices_to_nodes, adj_matrix, batch_size=64)
        g1_results[length_range] = sum(sl) / len(sl)
        print(f'  SR = {g1_results[length_range]:.3f}')

In [ ]:
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

plt.style.use("seaborn-v0_8")
plt.rcParams['axes.facecolor'] = '#f5f5f5'
plt.rcParams.update({'font.size': 11})

# All length groups present in either result set
all_groups = sorted(set(list(g1_results.keys()) + list(g2_results.keys())))
boundary_idx = sum(1 for g in all_groups if g[1] <= max_train_length)

# Build bar arrays:
#   y      = Same Map, Longer Length (G1, only for longer groups)
#   y_map2 = New Maps (G2, all groups — blue for ID, red_light for longer)
y = np.array([g1_results.get(g, 0) for g in all_groups])
y_map2 = np.array([g2_results.get(g, 0) for g in all_groups])

fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))

# Colors from coolwarm
cmap = plt.cm.coolwarm
blue_light = cmap(0.35)
red_light  = cmap(0.7)
red_dark   = cmap(0.99)

bar_width = 0.35
x_pos = np.arange(len(all_groups))

# Color each bar by segment
colors_g1 = [blue_light if i < boundary_idx else red_dark  for i in range(len(all_groups))]
colors_g2 = [blue_light if i < boundary_idx else red_light for i in range(len(all_groups))]

# For ID-length: only show one bar (G2 = spatial transfer, which is the blue bar)
# For longer: show two bars side by side (G1 = same map red_dark, G2 = new maps red_light)
bars1 = ax.bar(x_pos + bar_width/2, y,      bar_width, color=colors_g1, alpha=0.9, linewidth=0.4)
bars2 = ax.bar(x_pos - bar_width/2, y_map2, bar_width, color=colors_g2, alpha=0.9, linewidth=0.4)

# Hide G1 bars for ID-length (they are 0 anyway since we didn't evaluate)
# The blue bars from G2 cover both sides for ID length

# Boundary line
ax.axvline(x=boundary_idx - 0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.8)

# Axes
ax.set_xticks(x_pos)
ax.set_xticklabels([f"({a},{b}]" for (a, b) in all_groups], fontsize=10, rotation=45, ha='right')
ax.set_xlabel("Length Group", fontsize=12, fontweight='bold')
ax.set_ylabel("Success Rate", fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.4, linestyle='--', linewidth=0.6)

# Legend
handle_sp      = Patch(facecolor=blue_light, edgecolor='none', alpha=0.9)
handle_sp_long = Patch(facecolor=red_light,  edgecolor='none', alpha=0.9)
handle_no_long = Patch(facecolor=red_dark,   edgecolor='none', alpha=0.9)

ax.legend([handle_sp, handle_sp_long, handle_no_long],
          ['New Maps, Seen Length', 'New Maps, Longer Length', 'Same Map, Longer Length'],
          handler_map={tuple: HandlerTuple(ndivide=None, pad=0.4)},
          loc='best', fontsize=9, frameon=False)

# ID / Longer annotations
ax.annotate('ID Length',
            xy=((boundary_idx - 0.75) / len(all_groups), 1.02), xycoords='axes fraction',
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.25", facecolor=blue_light, alpha=0.25, edgecolor='none'))
ax.annotate('Longer Length',
            xy=((boundary_idx + (len(all_groups) - boundary_idx) / 2) / len(all_groups), 1.02),
            xycoords='axes fraction', ha='center', va='bottom', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.25", facecolor=red_light, alpha=0.25, edgecolor='none'))

plt.tight_layout()
plt.show()

## Table 1: Composition Analysis of Length Scaling Failure

Decompose long-path failures into hardness accumulation vs recursive instability.

In [ ]:
# For Table 1, we need to evaluate subpaths of failed long paths
# Pick length groups (20,30] and (30,40]
G = nx.from_numpy_array(adj_matrix)

for length_range in [(20, 30), (30, 40)]:
    test_pairs = G1_length_buckets.get(length_range, [])
    if not test_pairs:
        print(f'No test pairs for {length_range}')
        continue
    
    print(f'\n=== Length group {length_range} ({len(test_pairs)} pairs) ===')
    
    # Evaluate full paths
    success_list = annotate_failure_cases(
        sft_model, sft_tokenizer, sft_hf_tokenizer, 'sft',
        test_pairs, nodes_to_indices, indices_to_nodes, adj_matrix,
        batch_size=64
    )
    pr_long = sum(success_list) / len(success_list)
    
    # Use the stored results for subpath success rate
    pr_sub = g1_results.get((1, 10), 0.95)
    if (10, 20) in g1_results:
        pr_sub = (pr_sub + g1_results[(10, 20)]) / 2
    
    pr_sub1_and_sub2 = pr_sub ** 2
    pr_long_given_both = pr_long / pr_sub1_and_sub2 if pr_sub1_and_sub2 > 0 else 0
    pr_long_not_both = pr_long - pr_long_given_both * pr_sub1_and_sub2
    
    print(f'Pr(Long) = {pr_long:.3f}')
    print(f'Pr(Sub) = {pr_sub:.3f}')
    print(f'Pr(Sub1 ^ Sub2) = {pr_sub1_and_sub2:.3f}')
    print(f'Pr(Long | Sub1 ^ Sub2) = {pr_long_given_both:.3f}')
    print(f'Pr(Long, not(Sub1 ^ Sub2)) = {max(0, pr_long_not_both):.3f}')

## Failure Case Visualization (Fig 17-20, Table 6)

In [ ]:
# Collect failure cases from SFT model on (10,20) length group
length_range = (10, 20)
test_pairs = G1_length_buckets.get(length_range, [])

if test_pairs:
    sft_failures = collect_failure_cases_from_model(
        sft_model, sft_tokenizer, sft_hf_tokenizer, 'sft',
        test_pairs, nodes_to_indices, indices_to_nodes, adj_matrix,
        batch_size=32, max_failures=20
    )
    
    # Classify error types
    error_types = defaultdict(int)
    for fc in sft_failures:
        error_types[fc['failure_reason']] += 1
    
    print(f'\nSFT Error Types for {length_range}:')
    total = sum(error_types.values())
    for reason, count in sorted(error_types.items(), key=lambda x: -x[1]):
        print(f'  {reason}: {count} ({count/total*100:.1f}%)')

In [ ]:
# Visualize first 3 failure cases
if sft_failures:
    for i, fc in enumerate(sft_failures[:3]):
        fig, ax = visualize_failure_case(
            fc, adj_matrix, indices_to_nodes,
            title=f'SFT Failure Case - Length Range {length_range}',
            figsize=(8, 6)
        )
        plt.show()

## (Optional) Load GRPO model for comparison

In [ ]:
# Uncomment to also load and evaluate GRPO model
# grpo_model, grpo_tokenizer, grpo_hf_tokenizer = load_model_for_inference(
#     mode='grpo',
#     coverage=coverage_max,
#     diversity=diversity_max,
#     pairs_idx=pairs_idx,
#     model_dir=model_dir,
#     indices_to_nodes=indices_to_nodes,
#     nodes_to_indices=nodes_to_indices,
#     adj_matrix=adj_matrix
# )
# print('GRPO model loaded')